In [ ]:
import pandas as pd
import re
import numpy as np


input_file = "doaj_articles.csv"
output_file = "doaj_articles_fixed.csv"


def clean_text(text):
    """
    Funkcja czyszcząca tekst:
    1. Spłaszcza rozjechane wiersze (usuwa entery i wielokrotne spacje).
    2. Usuwa tagi HTML (<...>).
    3. Usuwa ewentualne ucięte tagi jats (bezpiecznie dla cudzysłowów).
    """
    if not isinstance(text, str):
        return ""
    
    # 1. Spłaszczanie (zamiana enterów i tabulatorów na spację)
    text = re.sub(r'\s+', ' ', text).strip()
    
    # 2. Usuwanie standardowych tagów (<i>, </i>, <p> itp.)
    text = re.sub(r'<[^>]+>', '', text)
    
    # 3. Dodatkowe zabezpieczenie na "dziwne" tagi (jak w CORE)
    text = re.sub(r'</?jats:\w+>?', '', text)
    
    return text

print(f"Wczytywanie pliku {input_file}...")
try:
    df = pd.read_csv(input_file, on_bad_lines='skip')
    print(f"Wczytano {len(df):,} wierszy.")
except Exception as e:
    print(f"Błąd krytyczny przy wczytywaniu: {e}")
    df = pd.DataFrame()

if not df.empty:
    # 1. Czyszczenie tekstów (Title, Abstract)
    print("Rozpoczynam czyszczenie tagów i spłaszczanie...")
    cols_to_clean = ['title', 'abstract', 'keywords', 'creator']
    
    for col in cols_to_clean:
        if col in df.columns:
            ## konwersja na string i czyszczenie
            df[col] = df[col].astype(str).apply(clean_text)
            ## zamiana pustych napisów i słowa 'nan' na prawdziwą pustkę (NaN)
            df[col] = df[col].replace(r'^\s*$', np.nan, regex=True)
            df[col] = df[col].replace('nan', np.nan)

    # 2. Uzupełnianie pustych keywords
    print("Uzupełnianie brakujących keywords wartościami z query...")
    if 'query' in df.columns and 'keywords' in df.columns:
        missing_count = df['keywords'].isna().sum()
        df['keywords'] = df['keywords'].fillna(df['query'])
        print(f"Uzupełniono {missing_count:,} pustych pól w 'keywords'.")

    # 3. Usuwanie śmieci (tylko gdy brak tytułu i abstraktu jednocześnie)
    print("Usuwanie pustych rekordów (bez tytułu i abstraktu)...")
    before_drop = len(df)
    ### how='all' oznacza: usuń wiersz tylko, jeśli wszystkie wskazane kolumny są puste (NaN)
    df = df.dropna(subset=['title', 'abstract'], how='all')
    after_drop = len(df)
    if before_drop > after_drop:
        print(f"Usunięto {before_drop - after_drop:,} kompletnie pustych wierszy.")

    # 4. Deduplikacja
    dedup_cols = ['title', 'abstract']
    print(f"Deduplikacja wg kolumn: {dedup_cols}...")
    
    initial_count = len(df)
    df = df.drop_duplicates(subset=dedup_cols, keep='first')
    final_count = len(df)
    
    print(f"Usunięto duplikatów: {initial_count - final_count:,}")
    print(f"Liczba unikalnych rekordów: {final_count:,}")

    # 5. Zapis
    print(f"Zapisywanie do {output_file}...")
    df.to_csv(output_file, index=False, encoding='utf-8')
    
    # podgląd wyniku
    print("\nPodgląd próbki danych")
    if 'year' in df.columns:
        print(df[['title', 'year']].head(3))
    else:
        print(df[['title']].head(3))
        
    if 'abstract' in df.columns and not df['abstract'].dropna().empty:
        print("\nPrzykładowy abstrakt (czysty):")
        print(df['abstract'].dropna().iloc[0][:200] + "...")
